# prep

In [1]:
import os

folder_path = "/Users/yerik/Music/try_new mp3/_1_YOGA"

file_names = [
    f for f in os.listdir(folder_path)
    if os.path.isfile(os.path.join(folder_path, f))
]

print(file_names)


['YOGA-7_DeepDub.txt', 'YOGA-9_ECSA-ARAB.txt', 'YOGA-3_Viby-FLOWS.txt', '.DS_Store', 'YOGA-2_Melo-no-words.txt', 'YOGA-10_ECSA-afronhouse.txt', 'YOGA-8_ecsa-remix.txt', 'YOGA-6_Bass-d&B.txt', 'YOGA-5_Chamanics.txt', 'YOGA-12_ECSA-FunkyStart.txt', 'YOGA-11_ECSA-hastaARRIBA.txt', 'YOGA-1_Meditate_astral.txt', 'YOGA-4_Bilateral_fxs.txt']


# copy files to new folder - from txt RK


In [1]:
# ============================================================
# === TXT (BOM DETECT) → AUTO FOLDER COPY → SAFE REWRITE TXT ==
# ============================================================

import os
import shutil
import pandas as pd
from tqdm import tqdm


def _copy_0701_txt_replace_SAFE_GET_df(txt_path,
                                      path_col='Location',
                                      sep='\t',
                                      overwrite_files=True,
                                      overwrite_txt=True):
    if not os.path.isfile(txt_path):
        raise FileNotFoundError(f"TXT not found: {txt_path}")

    # -------- BOM detect encoding --------
    with open(txt_path, 'rb') as f:
        head = f.read(4)

    if head.startswith(b'\xff\xfe') or head.startswith(b'\xfe\xff'):
        enc = 'utf-16'
    elif head.startswith(b'\xef\xbb\xbf'):
        enc = 'utf-8-sig'
    else:
        enc = None

    # -------- read txt --------
    enc_try = ([enc] if enc else []) + ['utf-16', 'utf-16-le', 'utf-8-sig', 'utf-8', 'latin-1']
    df = None
    used_enc = None
    read_errors = []

    for e in enc_try:
        if e is None:
            continue
        try:
            df = pd.read_csv(txt_path, sep=sep, encoding=e, dtype=str, engine='python')
            used_enc = e
            break
        except Exception as ex:
            read_errors.append((e, str(ex)))

    if df is None:
        raise UnicodeError(f"Failed reading TXT. Tried encodings: {read_errors}")

    df.columns = df.columns.str.strip()

    if path_col not in df.columns:
        raise ValueError(f"Column '{path_col}' not found. Columns: {list(df.columns)}")

    # -------- auto destination folder --------
    txt_dir = os.path.dirname(txt_path)
    txt_base = os.path.splitext(os.path.basename(txt_path))[0]
    dest_folder = os.path.join(txt_dir, txt_base)
    os.makedirs(dest_folder, exist_ok=True)

    # -------- sanitize paths --------
    def _clean_path(p):
        if p is None:
            return ""
        p = str(p)
        return p.replace('\ufeff', '').replace('\r', '').strip().strip('"').strip("'")

    # preserve original paths ALWAYS (prevents self-sabotage on second run)
    if 'Original_Location' not in df.columns:
        df['Original_Location'] = df[path_col]

    src_list = df['Original_Location'].fillna('').astype(str).tolist()

    new_paths = []
    status = []
    errors = []

    print(f"\nTXT  → {txt_path}")
    print(f"ENC  → {used_enc}")
    print(f"DEST → {dest_folder}")
    print(f"ROWS → {len(df)}\n")

    for src in tqdm(src_list, desc='Copying files', unit='file'):
        try:
            src = _clean_path(src)

            if (not src) or (not os.path.isfile(os.path.expanduser(src))):
                new_paths.append(None)
                status.append('missing')
                errors.append('not found')
                continue

            src2 = os.path.expanduser(src)
            fname = os.path.basename(src2)
            dst = os.path.join(dest_folder, fname)

            if overwrite_files or (not os.path.exists(dst)):
                shutil.copy2(src2, dst)

            new_paths.append(dst)
            status.append('copied')
            errors.append('')

        except Exception as e:
            new_paths.append(None)
            status.append('error')
            errors.append(str(e))

    # Only update Location when we actually copied successfully
    df['copy_status'] = status
    df['copy_error'] = errors

    df[path_col] = [
        new_paths[i] if status[i] == 'copied' else _clean_path(df.loc[i, 'Original_Location'])
        for i in range(len(df))
    ]

    # -------- overwrite txt in place --------
    if overwrite_txt:
        tmp_path = txt_path + ".__tmp__"
        out_enc = used_enc if used_enc else 'utf-16'

        df.to_csv(tmp_path, sep=sep, index=False, encoding=out_enc, lineterminator='\n')
        os.replace(tmp_path, txt_path)
        print(f"\nUPDATED TXT OVERWRITTEN → {txt_path}\n")

    return df


In [2]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!

TXT_PATH = '/Users/yerik/Downloads/______mp3_26/YOGA-4_Bilateral_fxs.txt'
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!


df = _copy_0701_txt_replace_SAFE_GET_df(
    txt_path=TXT_PATH,
    path_col='Location',
    sep='\t',
    overwrite_files=True,
    overwrite_txt=True
)



TXT  → /Users/yerik/Downloads/______mp3_26/YOGA-4_Bilateral_fxs.txt
ENC  → utf-16
DEST → /Users/yerik/Downloads/______mp3_26/YOGA-4_Bilateral_fxs
ROWS → 16



Copying files: 100%|████████████████████████████████████████████████████████| 16/16 [00:00<00:00, 20.13file/s]


UPDATED TXT OVERWRITTEN → /Users/yerik/Downloads/______mp3_26/YOGA-4_Bilateral_fxs.txt



# rename 

In [3]:
# ============================================================
# ====== FIX LOCATION PATHS (BOM/CR/QUOTES) + RENAME ==========
# ============================================================

import os
import pandas as pd
from tqdm import tqdm
from datetime import datetime


def _rename_0701_fixpaths_GET_df_from_txt(
    txt_path,
    custom_artist="DJ_Selphi",
    custom_genre="Salsa",
    custom_label="Bachata",
    custom_release_date="",
    custom_purchase_date="",
    sep="\t"
):
    tqdm.pandas()

    # --- BOM detect read (same logic as before, no guessing) ---
    with open(txt_path, "rb") as f:
        head = f.read(4)

    if head.startswith(b"\xff\xfe") or head.startswith(b"\xfe\xff"):
        enc = "utf-16"
    elif head.startswith(b"\xef\xbb\xbf"):
        enc = "utf-8-sig"
    else:
        enc = "utf-16"  # most DJ exports

    df = pd.read_csv(
        txt_path,
        sep=sep,
        encoding=enc,
        engine="python",
        on_bad_lines="skip",
        dtype=str
    )

    df.columns = df.columns.str.strip()

    # ---------- helpers ----------
    def _clean_str(s):
        if s is None:
            return ""
        s = str(s)
        # strip BOM + whitespace + CR
        return s.replace("\ufeff", "").replace("\r", "").strip().strip('"').strip("'")

    def _clean_filename_token(s):
        return (
            _clean_str(s)
            .replace(" ", "_").replace("/", "___").replace(",", "_")
            .replace("(", "").replace(")", "").replace("!", "")
            .replace("&", "and").replace("’", "").replace("'", "")
            .replace("¿", "").replace("¡", "").replace(":", "")
            .replace(";", "").strip()
        )

    def extract_mix(title):
        t = _clean_str(title).lower()
        if ("remix" in t) or ("mix" in t):
            return _clean_filename_token(title)
        return "original"

    def _parse_date_yyyymmdd(val):
        dt = pd.to_datetime(_clean_str(val), errors="coerce")
        return dt.strftime("%Y_%m_%d") if pd.notna(dt) else "NA"

    def _resolve_existing_path(p):
        """
        1) Try as-is
        2) Try expanding ~
        3) If still missing, try locating by filename in same folder as TXT’s auto-folder
           (this helps if Location column is stale but files were copied)
        """
        p = _clean_str(p)
        if not p:
            return None

        p2 = os.path.expanduser(p)
        if os.path.isfile(p2):
            return p2

        # fallback: try find by basename in the auto-folder created from TXT name
        txt_dir = os.path.dirname(txt_path)
        txt_base = os.path.splitext(os.path.basename(txt_path))[0]
        autofolder = os.path.join(txt_dir, txt_base)

        fname = os.path.basename(p2)
        cand = os.path.join(autofolder, fname)
        if os.path.isfile(cand):
            return cand

        return None

    def format_filename(row):
        title = _clean_filename_token(row.get("Track Title", ""))[:25]
        remix = extract_mix(row.get("Track Title", ""))

        artist_val = _clean_filename_token(custom_artist)[:25] if custom_artist else _clean_filename_token(row.get("Artist", ""))[:25]
        genre_val = _clean_filename_token(custom_genre) if custom_genre else _clean_filename_token(row.get("Genre", ""))
        label_val = _clean_filename_token(custom_label) if custom_label else _clean_filename_token(row.get("Label", ""))

        release_date_val = custom_release_date if custom_release_date else _parse_date_yyyymmdd(row.get("Release Date", ""))
        key = _clean_filename_token(row.get("Key", "NA")) or "NA"

        bpm_raw = _clean_str(row.get("BPM", ""))
        try:
            bpm = str(int(round(float(bpm_raw))))
        except Exception:
            bpm = "NA"

        purchase_date_val = custom_purchase_date if custom_purchase_date else _parse_date_yyyymmdd(row.get("Date Added", datetime.today()))

        # extension comes from Location after resolution (safer), but fallback to .mp3
        return title, artist_val, remix, key, bpm, genre_val, label_val, release_date_val, purchase_date_val

    # ---------- main loop ----------
    new_paths = []
    missing_rows = 0

    for i, row in tqdm(df.iterrows(), total=len(df), desc="Renaming files"):
        original_path_raw = row.get("Location", "")
        original_path = _resolve_existing_path(original_path_raw)

        if not original_path:
            missing_rows += 1
            new_paths.append(None)
            continue

        title, artist_val, remix, key, bpm, genre_val, label_val, release_date_val, purchase_date_val = format_filename(row)
        ext = os.path.splitext(original_path)[1] or ".mp3"

        new_filename = (
            f"TRkw_{title}_ARkw_{artist_val}_MXkw_{remix}_KYkw_{key}_"
            f"BPkw_{bpm}_GNkw_{genre_val}_LBkw_{label_val}_RYkw_{release_date_val}_"
            f"PYkw_{purchase_date_val}{ext}"
        )

        if len(new_filename) > 240:
            new_filename = new_filename[:230] + ext

        new_path = os.path.join(os.path.dirname(original_path), new_filename)

        try:
            os.rename(original_path, new_path)
            new_paths.append(new_path)
        except Exception:
            new_paths.append(None)

    df["Renamed_Path"] = new_paths

    if missing_rows:
        print(f"\n⚠️ Missing files for {missing_rows} rows.")
        print("Most likely: Location has stale paths, hidden characters, or copy_status != copied.\n")

    return df


In [4]:
txt_path =TXT_PATH

In [5]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!


df = _rename_0701_fixpaths_GET_df_from_txt(
    txt_path=TXT_PATH,
    custom_artist="",
    custom_genre="",
    custom_label=""
)


Renaming files: 100%|████████████████████████████████████████████████████████| 16/16 [00:00<00:00, 838.76it/s]


# Transform to AIFF


# starts

In [6]:
# -----######-----###### CORE IMPORTABLE FUNCTION (All → AIFF 44.1/16/Stereo + Tags + Verify) -----######-----###### #
import os, sys, shutil, subprocess, tempfile
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

# Tagging
from mutagen import File as MutaFile
from mutagen.aiff import AIFF
from mutagen.id3 import (
    ID3, ID3NoHeaderError, ID3BadUnsynchData,
    TIT2, TPE1, TPE2, TALB, TCON, TDRC, TRCK, TPOS, COMM, TBPM, TKEY,
    TPUB, TSRC, TPE3, TCOM, TENC, APIC, CTOC, CHAP
)

# -------------------- helpers (no ASCII banner for sub-fns) -------------------- #
def _safe_get_first(d, key):
    if d is None: return None
    v = d.get(key)
    if v is None: return None
    if isinstance(v, (list, tuple)):
        return v[0] if v else None
    return v

def _as_int_pair(text):
    if not text: return None, None
    s = str(text)
    if '/' in s:
        a,b = s.split('/',1)
        return (a.strip() or None), (b.strip() or None)
    return (s.strip() or None), None

def _ensure_id3(aiff_path):
    a = AIFF(aiff_path)
    if a.tags is None:
        a.add_tags()
    return a

def _copy_id3_frames(src_id3, dst_id3):
    # Copy common frames + chapters/artwork; ignore oddities that fail to serialize.
    for frame in list(src_id3.values()):
        try:
            if isinstance(frame, APIC):
                dst_id3.add(APIC(encoding=frame.encoding, mime=frame.mime, type=frame.type, desc=frame.desc, data=frame.data))
            elif isinstance(frame, COMM):
                dst_id3.add(COMM(encoding=frame.encoding, lang=frame.lang, desc=frame.desc, text=frame.text))
            elif isinstance(frame, (TIT2, TPE1, TPE2, TALB, TCON, TDRC, TRCK, TPOS, TBPM, TKEY, TPUB, TSRC, TPE3, TCOM, TENC)):
                dst_id3.add(type(frame)(encoding=frame.encoding, text=frame.text))
            elif isinstance(frame, (CTOC, CHAP)):
                dst_id3.add(frame)
            else:
                # Pass through for other safe T* frames
                dst_id3.add(frame)
        except Exception:
            # Skip non-serializable frames without killing the run
            pass

def _map_generic_to_id3(vtags, dst_id3, pictures=None):
    # Generic (Vorbis/FLAC/WAV INFO) → ID3
    title   = _safe_get_first(vtags, "title")
    artist  = _safe_get_first(vtags, "artist")
    album   = _safe_get_first(vtags, "album")
    albumartist = _safe_get_first(vtags, "albumartist") or _safe_get_first(vtags, "album artist")
    genre   = _safe_get_first(vtags, "genre")
    date    = _safe_get_first(vtags, "date") or _safe_get_first(vtags, "year")
    comment = _safe_get_first(vtags, "comment") or _safe_get_first(vtags, "description")
    bpm     = _safe_get_first(vtags, "bpm")
    key_    = _safe_get_first(vtags, "initialkey") or _safe_get_first(vtags, "key")
    label   = _safe_get_first(vtags, "label") or _safe_get_first(vtags, "publisher")
    isrc    = _safe_get_first(vtags, "isrc")
    remixer = _safe_get_first(vtags, "remixer")
    composer= _safe_get_first(vtags, "composer")
    encoder = _safe_get_first(vtags, "encoder") or _safe_get_first(vtags, "encodedby") or _safe_get_first(vtags, "encoded_by")
    trk     = _safe_get_first(vtags, "tracknumber")
    dsk     = _safe_get_first(vtags, "discnumber")

    if title:   dst_id3.add(TIT2(encoding=3, text=str(title)))
    if artist:  dst_id3.add(TPE1(encoding=3, text=str(artist)))
    if album:   dst_id3.add(TALB(encoding=3, text=str(album)))
    if albumartist: dst_id3.add(TPE2(encoding=3, text=str(albumartist)))
    if genre:   dst_id3.add(TCON(encoding=3, text=str(genre)))
    if date:    dst_id3.add(TDRC(encoding=3, text=str(date)))
    if comment: dst_id3.add(COMM(encoding=3, lang="eng", desc="", text=str(comment)))
    if bpm:     dst_id3.add(TBPM(encoding=3, text=str(bpm)))
    if key_:    dst_id3.add(TKEY(encoding=3, text=str(key_)))
    if label:   dst_id3.add(TPUB(encoding=3, text=str(label)))
    if isrc:    dst_id3.add(TSRC(encoding=3, text=str(isrc)))
    if remixer: dst_id3.add(TPE3(encoding=3, text=str(remixer)))
    if composer:dst_id3.add(TCOM(encoding=3, text=str(composer)))
    if encoder: dst_id3.add(TENC(encoding=3, text=str(encoder)))

    if trk:
        n, d = _as_int_pair(trk)
        if n or d:
            dst_id3.add(TRCK(encoding=3, text=[f"{n or ''}/{d or ''}".strip('/')]))
    if dsk:
        n, d = _as_int_pair(dsk)
        if n or d:
            dst_id3.add(TPOS(encoding=3, text=[f"{n or ''}/{d or ''}".strip('/')]))

    if pictures:
        for pic in pictures:
            try:
                dst_id3.add(APIC(encoding=3, mime=getattr(pic, "mime", None) or "image/jpeg", type=3, desc=u"", data=getattr(pic, "data", b"")))
            except Exception:
                pass

def _copy_all_tags_to_aiff(src_path, aiff_path):
    """
    After encoding, write a clean ID3 tag set into the AIFF.
    Priority:
      1) If source has ID3 → copy frames
      2) Else, map Vorbis/FLAC/WAV INFO → ID3; copy artwork when possible
    """
    dst_aiff = _ensure_id3(aiff_path)
    dst_id3 = dst_aiff.tags

    src = MutaFile(src_path)
    if src is None:
        dst_aiff.save()
        return

    # Direct ID3 → ID3
    try:
        src_id3 = getattr(src, "tags", None)
        if isinstance(src_id3, ID3) or (src_id3 and any(k.startswith("T") or k in ("APIC","COMM","CTOC","CHAP") for k in src_id3.keys())):
            try:
                _copy_id3_frames(src_id3, dst_id3)
                dst_aiff.save()
                return
            except Exception:
                pass
    except Exception:
        pass

    # Generic mapping (Vorbis/FLAC/WAV INFO, MP4 atoms won't map fully)
    vtags = getattr(src, "tags", {}) or {}
    pictures = []
    try:
        # FLAC: embedded pictures
        if hasattr(src, "pictures") and getattr(src, "pictures", None):
            pictures = src.pictures
        elif hasattr(src, "tags") and "METADATA_BLOCK_PICTURE" in src.tags:
            pictures = []  # base64 case skipped (mutagen handles some variants)
    except Exception:
        pictures = []

    try:
        _map_generic_to_id3(vtags, dst_id3, pictures=pictures)
    except Exception:
        pass

    dst_aiff.save()

def _verify_aiff_ok(aiff_path):
    try:
        t = AIFF(aiff_path)
        _ = t.info.length  # raises if broken
        return True, None
    except Exception as e:
        return False, str(e)

def _exts_casefold(exts):
    # normalize to a case-insensitive set, include upper/lower/Title variants
    s = set()
    for e in exts or []:
        if not e: continue
        ee = e if e.startswith(".") else "."+e
        base = ee.lower()
        s.add(base)
        s.add(base.upper())
        s.add(base.capitalize())
    return s

def _ffmpeg_encode_to_aiff(src_path, dst_path):
    """
    Robust ffmpeg call:
    - force AIFF PCM 16-bit big-endian, 44.1kHz, stereo
    - strip container metadata (we'll write fresh ID3 next)
    - disable video/subs
    - choose first audio stream explicitly
    """
    cmd = [
        "ffmpeg",
        "-hide_banner", "-loglevel", "error",
        "-y",
        "-i", str(src_path),
        "-map", "0:a:0",
        "-vn", "-sn",
        "-ar", "44100",
        "-ac", "2",
        "-c:a", "pcm_s16be",
        "-map_metadata", "-1",
        str(dst_path)
    ]
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return p.returncode == 0

# -----######-----###### CORE IMPORTABLE FUNCTION (per-file) -----######-----###### #
def _aiff_0109_onefile_GET_status_path(
    file_in,
    out_dir=None
):
    """
    Convert a single file → AIFF (44.1kHz / 16-bit / stereo), always re-encode (even AIFF).
    Returns (status, path_out). status in {"ok", "broken", "fail"}.
    """
    src_path = Path(file_in)
    if out_dir is None:
        out_dir = src_path.parent
    else:
        out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)

    dst_path = out_dir / (src_path.stem + ".aiff")

    # Atomic write via temp file to avoid half-written outputs
    with tempfile.TemporaryDirectory() as td:
        tmp_path = Path(td) / (src_path.stem + ".aiff")

        ok = _ffmpeg_encode_to_aiff(src_path, tmp_path)
        if not ok or not tmp_path.exists():
            return "fail", None

        # Write tags/artwork (best-effort; never fatal)
        try:
            _copy_all_tags_to_aiff(src_path, tmp_path)
        except Exception:
            pass

        # Verify playable
        v_ok, v_err = _verify_aiff_ok(tmp_path)
        if not v_ok:
            # Move the broken file out with suffix
            broken_dst = dst_path.with_stem(dst_path.stem + "_BROKEN")
            try:
                broken_dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(tmp_path), str(broken_dst))
            except Exception:
                pass
            return "broken", broken_dst if broken_dst.exists() else dst_path

        # All good: move into place
        dst_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(tmp_path), str(dst_path))

    return "ok", dst_path

# -----######-----###### CORE IMPORTABLE FUNCTION (batch) -----######-----###### #
def _aiff_0109_all2aiff_GET_summary(
    root_folder,
    out_root=None,
    audio_extensions=None,
    dry_run="n",
    src_action="keep",          # "keep" | "move" | "trash"
    src_action_move_dir=None,   # required if src_action == "move"
    overwrite="n"               # "y" to overwrite existing AIFF at dst, else skip creating duplicate
):
    """
    Recursively convert *major* audio types to AIFF (CDJ-safe 44.1/16/stereo) with tags & artwork.
    - Always re-encodes, even for AIFF sources (to guarantee target spec)
    - Mirrors folder structure under out_root (if provided)
    - Case-insensitive extension handling
    - macOS junk skipped
    - TQM progress bar + compact summary
    - Post-success actions on sources: keep | move | trash

    Returns:
      summary dict with counts and lists per status.
    """
    root = Path(root_folder)

    # If user doesn't pass, we include a broad set of common types
    if audio_extensions is None:
        audio_extensions = [
            ".flac", ".wav", ".mp3", ".aiff", ".aif",
            ".m4a", ".aac", ".alac", ".ogg", ".oga", ".wv", ".aifc"
        ]
    exts_all = _exts_casefold(audio_extensions)

    # Collect candidates (skip macOS junk)
    all_files = [
        p for p in root.rglob("*")
        if p.is_file()
        and not p.name.startswith("._")
        and p.name != ".DS_Store"
        and (p.suffix in exts_all)
    ]

    # Compute outputs + decide skips
    targets = []
    for src_path in all_files:
        rel = src_path.relative_to(root)
        out_dir = (Path(out_root) / rel.parent) if out_root else src_path.parent
        dst_path = out_dir / (src_path.stem + ".aiff")

        if dry_run.lower().startswith("y"):
            targets.append((src_path, out_dir, dst_path, "todo"))
        else:
            # If overwrite == 'n' and AIFF already exists in destination, skip re-creating file
            if dst_path.exists() and not overwrite.lower().startswith("y"):
                # Still *verify* later if the existing AIFF matches spec? We assume OK for speed.
                # If you want forced re-encode, set overwrite='y'.
                continue
            targets.append((src_path, out_dir, dst_path, "todo"))

    # ----- TQM BAR -----
    pbar = tqdm(total=len(targets), desc="TQM | All → AIFF 44.1/16/stereo", unit="file")

    stats = {
        "ok": 0, "broken": 0, "fail": 0,
        "skipped_existing": 0,
        "total_scanned": len(all_files),
        "total_planned": len(targets),
        "ok_paths": [], "broken_paths": [], "fail_paths": [], "skipped_paths": []
    }

    # If dry-run: just preview names and return summary
    if dry_run.lower().startswith("y"):
        for (src_path, out_dir, dst_path, _) in targets:
            pbar.set_postfix_str(f"DRY-RUN → {src_path.name}")
            pbar.update(1)
            stats["skipped_paths"].append(str(src_path))
        pbar.close()
        return stats

    # Convert
    for (src_path, out_dir, dst_path, _) in targets:
        # If we got here and the destination exists but overwrite == 'n', mark skipped
        if dst_path.exists() and not overwrite.lower().startswith("y"):
            stats["skipped_existing"] += 1
            stats["skipped_paths"].append(str(dst_path))
            pbar.set_postfix_str(f"SKIP (exists): {dst_path.name}")
            pbar.update(1)
            continue

        status, outp = _aiff_0109_onefile_GET_status_path(src_path, out_dir=out_dir)
        if status == "ok":
            stats["ok"] += 1
            stats["ok_paths"].append(str(outp))
            pbar.set_postfix_str(f"OK: {src_path.name}")
            # Post-success action on source
            try:
                if src_action == "trash":
                    # move to user Trash if available; fallback to unlink
                    try:
                        from send2trash import send2trash
                        send2trash(str(src_path))
                    except Exception:
                        src_path.unlink(missing_ok=True)
                elif src_action == "move":
                    if not src_action_move_dir:
                        raise ValueError("src_action_move_dir is required when src_action='move'")
                    dst_dir_move = Path(src_action_move_dir); dst_dir_move.mkdir(parents=True, exist_ok=True)
                    shutil.move(str(src_path), str(dst_dir_move / src_path.name))
                else:
                    pass  # keep
            except Exception:
                # Non-fatal; keep going
                pass

        elif status == "broken":
            stats["broken"] += 1
            stats["broken_paths"].append(str(outp))
            pbar.set_postfix_str(f"BROKEN: {src_path.name}")
        else:
            stats["fail"] += 1
            stats["fail_paths"].append(str(src_path))
            pbar.set_postfix_str(f"FAIL: {src_path.name}")

        pbar.update(1)

    pbar.close()

    stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(
        f"[{stamp}] === SUMMARY ===\n"
        f"Total scanned:     {stats['total_scanned']}\n"
        f"Planned to convert:{stats['total_planned']}\n"
        f"Converted OK:      {stats['ok']}\n"
        f"Broken (tagged):   {stats['broken']}\n"
        f"Failed:            {stats['fail']}\n"
        f"Skipped (exists):  {stats['skipped_existing']}\n"
    )
    return stats


In [7]:
# ! CHANGE THESE AS NEEDED
root_folder = "/Users/yerik/Downloads/______mp3_26/YOGA-1_Meditate_astral"


In [8]:
out_root    = None  # or e.g. "/Volumes/HD_back_UP/ALL_MUSIC/_AIFF_OUT"

# Externalized extensions (case-insensitive will be auto-handled)
audio_extensions = [".flac", ".wav", ".mp3", ".aiff", ".aif", ".m4a", ".aac", ".alac", ".ogg", ".oga", ".wv", ".aifc"]

# Action knobs
dry_run     = "n"           # "y" to preview only
overwrite   = "n"           # "y" to force re-encode even if dst exists
src_action  = "keep"        # "keep" | "move" | "trash"
move_dir    = ""  # required if src_action=="move"

summary = _aiff_0109_all2aiff_GET_summary(
    root_folder=root_folder,
    out_root=out_root,
    audio_extensions=audio_extensions,
    dry_run=dry_run,
    src_action=src_action,
    src_action_move_dir=move_dir,
    overwrite=overwrite
)


TQM | All → AIFF 44.1/16/stereo: 100%|█| 10/10 [00:02<00:00,  3.68file/s, OK: TRkw__ms_OLAS_van_ARkw_DJ_Selphi

[2026-01-07 20:23:49] === SUMMARY ===
Total scanned:     10
Planned to convert:10
Converted OK:      10
Broken (tagged):   0
Failed:            0
Skipped (exists):  0



# erase originals after checking 

In [9]:
# -----######-----###### CORE IMPORTABLE FUNCTION (Erase everything but AIFF) -----######-----###### #
import os
from pathlib import Path
from tqdm import tqdm

def _cleanup_2208_keepaiff_GET_removed_files(root_folder, dry_run="n"):
    """
    Recursively erase everything but AIFF (.aiff/.aif) files.
    Skips system junk (.DS_Store, ._*).
    
    Inputs:
      root_folder : str/Path → folder to clean
      dry_run     : "y" = preview only, "n" = actually delete
    
    Returns:
      dict summary with counts
    """
    root = Path(root_folder)

    # Collect all files
    all_files = [p for p in root.rglob("*") if p.is_file()]
    # Keep only those NOT AIFF
    targets = [
        p for p in all_files
        if p.suffix.lower() not in (".aiff", ".aif")
        and not p.name.startswith("._")
        and p.name != ".DS_Store"
    ]

    # Progress bar
    pbar = tqdm(total=len(targets), desc="TQM • Cleaning non-AIFF files", unit="file")

    removed, skipped = 0, 0
    for f in targets:
        if dry_run.lower().startswith("y"):
            pbar.set_postfix_str(f"DRY-RUN: would remove {f.name}")
            skipped += 1
        else:
            try:
                f.unlink()
                removed += 1
                pbar.set_postfix_str(f"Removed {f.name}")
            except Exception as e:
                skipped += 1
                pbar.set_postfix_str(f"⚠️ Skip {f.name}: {e}")
        pbar.update(1)

    pbar.close()

    summary = {
        "total_files": len(all_files),
        "removed": removed,
        "skipped": skipped,
        "kept_aiff": len(all_files) - len(targets),
    }

    print(
        f"Done cleanup.\n"
        f" • Total files scanned: {summary['total_files']}\n"
        f" • Removed: {summary['removed']}\n"
        f" • Skipped (errors/dry-run): {summary['skipped']}\n"
        f" • Kept AIFF: {summary['kept_aiff']}\n"
    )
    return summary


In [10]:
_cleanup_2208_keepaiff_GET_removed_files(root_folder, dry_run="n")


TQM • Cleaning non-AIFF files: 100%|█| 10/10 [00:00<00:00, 638.65file/s, Removed TRkw__ms_OLAS_van_ARkw_DJ_Sel

Done cleanup.
 • Total files scanned: 20
 • Removed: 10
 • Skipped (errors/dry-run): 0
 • Kept AIFF: 10



{'total_files': 20, 'removed': 10, 'skipped': 0, 'kept_aiff': 10}

# Transform to MP3

In [2]:
# ============================================#
# -----######-----######  CORE FUNCTION  -----#
# _mp3_0701_aiffwav2mp3tagname_GET_df_results
# ============================================#

import os
import re
import subprocess
from pathlib import Path

import pandas as pd
from tqdm import tqdm

from mutagen import File as MFile
from mutagen.id3 import (
    ID3, ID3NoHeaderError,
    TIT2, TPE1, TALB, TCON, TDRC, TRCK, COMM, TXXX, TPUB, TPE4
)
from mutagen.mp3 import MP3


# -----------------------------#
# helpers (no ASCII art)
# -----------------------------#

def _ensure_ffmpeg():
    try:
        subprocess.run(["ffmpeg", "-version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        return True
    except Exception:
        return False

def _is_hidden_or_junk(name):
    return name.startswith("._") or name.startswith(".DS") or name.startswith(".")

def _first(v):
    if v is None:
        return None
    if isinstance(v, (list, tuple)) and len(v) > 0:
        return v[0]
    return v

def _clean_text(x):
    if x is None:
        return None
    s = str(x).strip()
    return s if s else None

def _norm_spaces(s):
    if s is None:
        return None
    s = re.sub(r"\s+", " ", str(s)).strip()
    return s if s else None

def _fs_safe(s):
    """
    Make string safe for filenames while keeping it readable.
    """
    if s is None:
        return None
    s = str(s)

    # kill path separators + risky chars
    s = s.replace("/", "-").replace("\\", "-")
    s = re.sub(r"[<>:\"|?*\n\r\t]+", " ", s)

    # collapse spaces/underscores
    s = re.sub(r"\s+", " ", s).strip()
    s = s.replace(" ", "_")
    s = re.sub(r"_+", "_", s).strip("_")

    return s if s else None

def _read_tags_any(path):
    """
    Best-effort tag extraction from AIFF/WAV:
    - Mutagen easy tags
    - ID3 frames (AIFF commonly has ID3)
    Returns dict + ID3 object (or None)
    """
    tags_easy = {}
    id3 = None

    try:
        mf = MFile(path, easy=True)
        if mf is not None and getattr(mf, "tags", None):
            for k, v in dict(mf.tags).items():
                tags_easy[str(k).lower()] = _first(v)
    except Exception:
        pass

    try:
        id3 = ID3(path)
    except ID3NoHeaderError:
        id3 = None
    except Exception:
        id3 = None

    return tags_easy, id3

def _pick(tags_easy, *keys):
    for k in keys:
        v = _clean_text(tags_easy.get(k))
        if v:
            return v
    return None

def _get_id3_text(id3_obj, frame_key):
    if id3_obj is None:
        return None
    try:
        fr = id3_obj.get(frame_key)
        if fr and getattr(fr, "text", None):
            return _clean_text(fr.text[0])
    except Exception:
        return None
    return None

def _get_txxx(id3_obj, desc_lower_set):
    if id3_obj is None:
        return None
    try:
        for fr in id3_obj.getall("TXXX"):
            d = str(getattr(fr, "desc", "")).strip().lower()
            if d in desc_lower_set:
                return _clean_text(_first(fr.text))
    except Exception:
        pass
    return None

def _extract_apic(id3_obj):
    if id3_obj is None:
        return None
    try:
        apics = id3_obj.getall("APIC")
        if apics:
            return apics[0]
    except Exception:
        return None
    return None

def _ffmpeg_to_mp3(src, dst, bitrate_kbps=320, sr_hz=44100, channels=2):
    """
    DJ-acceptable:
    - CBR 320k
    - 44.1 kHz
    - stereo
    """
    cmd = [
        "ffmpeg", "-y",
        "-i", str(src),
        "-vn",
        "-ac", str(int(channels)),
        "-ar", str(int(sr_hz)),
        "-b:a", f"{int(bitrate_kbps)}k",
        "-map_metadata", "0",
        str(dst)
    ]
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    return proc.returncode == 0, proc.stderr[-2000:]


def _build_name_from_tags(stem_fallback, tags_easy, id3_src, n):
    """
    Output filename pattern (your global target format):
    _n#_TRkw_{track}_ARkw_{artist}_MXkw_{mix}_KYkw_{key}_BPkw_{bpm}_
    GNkw_{genre}_RMkw_{remixer}_LBkw_{label}_RYkw_{yyyy_mm_dd}_PYkw_{yyyy_mm_dd}.mp3

    Rule: if a field is missing, SKIP that chunk (no fake defaults).
    """
    # core pulls
    title  = _pick(tags_easy, "title") or _get_id3_text(id3_src, "TIT2") or stem_fallback
    artist = _pick(tags_easy, "artist", "artists") or _get_id3_text(id3_src, "TPE1")
    genre  = _pick(tags_easy, "genre") or _get_id3_text(id3_src, "TCON")
    label  = _pick(tags_easy, "label", "publisher", "organization") or _get_id3_text(id3_src, "TPUB")

    # extra pulls (often TXXX)
    bpm = _pick(tags_easy, "bpm") or _get_txxx(id3_src, {"bpm", "tbpm"})
    key = _pick(tags_easy, "initialkey", "key") or _get_txxx(id3_src, {"initialkey", "key", "tkey"})

    # remixer / mix name (best-effort; many libs store these inconsistently)
    remixer = _pick(tags_easy, "remixer") or _get_id3_text(id3_src, "TPE4") or _get_txxx(id3_src, {"remixer", "mixremixer"})
    mix_name = _pick(tags_easy, "mix", "mixname", "version") or _get_txxx(id3_src, {"mix", "mixname", "version"})

    # dates (only if you have them already)
    rel_date = _pick(tags_easy, "date", "originaldate") or _get_id3_text(id3_src, "TDRC")
    pur_date = _pick(tags_easy, "purchase_date", "purchasedate") or _get_txxx(id3_src, {"purchase_date", "purchasedate", "dateadded"})

    def date_to_yyyymmdd(d):
        if not d:
            return None
        s = str(d).strip()
        # accept "YYYY-MM-DD", "YYYY/MM/DD", "YYYY.MM.DD", or "YYYY"
        m = re.search(r"(\d{4})[^\d]?(\d{2})?[^\d]?(\d{2})?", s)
        if not m:
            return None
        y = m.group(1)
        mo = m.group(2) if m.group(2) else "00"
        da = m.group(3) if m.group(3) else "00"
        return f"{y}_{mo}_{da}"

    rel_ymd = date_to_yyyymmdd(rel_date)
    pur_ymd = date_to_yyyymmdd(pur_date)

    # sanitize for filesystem
    title_s   = _fs_safe(_norm_spaces(title))
    artist_s  = _fs_safe(_norm_spaces(artist))
    mix_s     = _fs_safe(_norm_spaces(mix_name))
    key_s     = _fs_safe(_norm_spaces(key))
    bpm_s     = _fs_safe(_norm_spaces(bpm))
    genre_s   = _fs_safe(_norm_spaces(genre))
    remixer_s = _fs_safe(_norm_spaces(remixer))
    label_s   = _fs_safe(_norm_spaces(label))

    parts = []
    parts.append(f"_n{int(n)}_")

    if title_s:   parts.append(f"TRkw_{title_s}_")
    if artist_s:  parts.append(f"ARkw_{artist_s}_")
    if mix_s:     parts.append(f"MXkw_{mix_s}_")
    if key_s:     parts.append(f"KYkw_{key_s}_")
    if bpm_s:     parts.append(f"BPkw_{bpm_s}_")
    if genre_s:   parts.append(f"GNkw_{genre_s}_")
    if remixer_s: parts.append(f"RMkw_{remixer_s}_")
    if label_s:   parts.append(f"LBkw_{label_s}_")
    if rel_ymd and rel_ymd != "0000_00_00":
        parts.append(f"RYkw_{rel_ymd}_")
    if pur_ymd and pur_ymd != "0000_00_00":
        parts.append(f"PYkw_{pur_ymd}_")

    fname = "".join(parts)
    fname = re.sub(r"_+", "_", fname)  # compress
    fname = fname.strip("_") + ".mp3"
    return fname


def _write_id3_mp3(mp3_path, tags_easy, id3_src):
    """
    Best-effort:
    - write common fields to MP3 ID3
    - preserve artwork when source has APIC
    - store BPM + INITIALKEY as TXXX (good interoperability)
    - store remixer as TPE4 if present
    """
    # load/create
    try:
        audio = MP3(mp3_path)
        id3 = audio.tags if audio.tags is not None else ID3()
    except Exception:
        id3 = ID3()

    title  = _pick(tags_easy, "title") or _get_id3_text(id3_src, "TIT2")
    artist = _pick(tags_easy, "artist", "artists") or _get_id3_text(id3_src, "TPE1")
    album  = _pick(tags_easy, "album") or _get_id3_text(id3_src, "TALB")
    genre  = _pick(tags_easy, "genre") or _get_id3_text(id3_src, "TCON")
    date   = _pick(tags_easy, "date", "originaldate", "year") or _get_id3_text(id3_src, "TDRC")
    track  = _pick(tags_easy, "tracknumber", "track", "trackno") or _get_id3_text(id3_src, "TRCK")

    comment = _pick(tags_easy, "comment", "comments", "description")
    if not comment and id3_src is not None:
        try:
            comms = id3_src.getall("COMM")
            if comms and getattr(comms[0], "text", None):
                comment = _clean_text(_first(comms[0].text))
        except Exception:
            pass

    label = _pick(tags_easy, "label", "publisher", "organization") or _get_id3_text(id3_src, "TPUB")
    bpm   = _pick(tags_easy, "bpm") or _get_txxx(id3_src, {"bpm", "tbpm"})
    key   = _pick(tags_easy, "initialkey", "key") or _get_txxx(id3_src, {"initialkey", "key", "tkey"})
    remixer = _pick(tags_easy, "remixer") or _get_id3_text(id3_src, "TPE4") or _get_txxx(id3_src, {"remixer", "mixremixer"})

    def set_frame(frame_id, frame_obj):
        try:
            id3.setall(frame_id, [frame_obj])
        except Exception:
            pass

    if title:
        set_frame("TIT2", TIT2(encoding=3, text=[title]))
    if artist:
        set_frame("TPE1", TPE1(encoding=3, text=[artist]))
    if album:
        set_frame("TALB", TALB(encoding=3, text=[album]))
    if genre:
        set_frame("TCON", TCON(encoding=3, text=[genre]))
    if date:
        set_frame("TDRC", TDRC(encoding=3, text=[str(date)]))
    if track:
        set_frame("TRCK", TRCK(encoding=3, text=[str(track)]))
    if comment:
        try:
            id3.setall("COMM", [COMM(encoding=3, lang="eng", desc="Comment", text=[comment])])
        except Exception:
            pass
    if label:
        set_frame("TPUB", TPUB(encoding=3, text=[label]))
    if remixer:
        set_frame("TPE4", TPE4(encoding=3, text=[remixer]))

    # BPM + KEY as TXXX
    if bpm:
        try:
            id3.add(TXXX(encoding=3, desc="BPM", text=[str(bpm)]))
        except Exception:
            pass
    if key:
        try:
            id3.add(TXXX(encoding=3, desc="INITIALKEY", text=[str(key)]))
        except Exception:
            pass

    # artwork copy
    apic = _extract_apic(id3_src)
    if apic is not None:
        try:
            id3.add(apic)
        except Exception:
            pass

    id3.save(mp3_path)


def _mp3_0701_aiffwav2mp3tagname_GET_df_results(
    folder_in,
    audio_extensions,
    bitrate_kbps=320,
    sr_hz=44100,
    channels=2,
    n_start=1,
    overwrite=False,
    keep_originals=True
):
    """
    What it does (properly):
    1) Walks folder_in recursively, finds AIFF/WAV (or whatever you pass)
    2) Converts each file to MP3 (DJ acceptable: 320k CBR / 44.1k / stereo)
    3) Writes MP3 into the SAME folder as source (unless you change that)
    4) Renames MP3 using your global naming format (TRkw/ARkw/...),
       AND always prefixes _n#_
       Even if conversion is skipped/fails, you still get a row in df.
    5) Preserves tags + artwork best-effort

    Returns df_results with full trace of what happened.
    """
    if not _ensure_ffmpeg():
        raise RuntimeError("ffmpeg not found. Install it (brew install ffmpeg) and try again.")

    folder_in = Path(folder_in)
    exts = {e.lower().lstrip(".") for e in (audio_extensions or [])}

    files = []
    for p in folder_in.rglob("*"):
        if not p.is_file():
            continue
        if _is_hidden_or_junk(p.name):
            continue
        if p.suffix.lower().lstrip(".") in exts:
            files.append(p)
    files = sorted(files)

    rows = []
    n = int(n_start)

    for src in tqdm(files, desc="TQM | AIFF/WAV -> MP3 + tags + naming", unit="file"):
        tags_easy, id3_src = _read_tags_any(src)

        # Build final filename in YOUR target format
        new_fname = _build_name_from_tags(
            stem_fallback=src.stem,
            tags_easy=tags_easy,
            id3_src=id3_src,
            n=n
        )
        dst = src.parent / new_fname

        row = {
            "n": n,
            "src_path": str(src),
            "src_ext": src.suffix.lower(),
            "dst_path": str(dst),
            "dst_file": new_fname,
            "status": None,
            "converted": False,
            "tags_written": False,
            "error": None,
        }

        try:
            if dst.exists() and not overwrite:
                row["status"] = "skip_exists"
                rows.append(row)
                n += 1
                continue

            ok, err_tail = _ffmpeg_to_mp3(
                src=src,
                dst=dst,
                bitrate_kbps=bitrate_kbps,
                sr_hz=sr_hz,
                channels=channels
            )

            if not ok or (not dst.exists()):
                row["status"] = "convert_failed"
                row["error"] = _clean_text(err_tail) or "ffmpeg failed"
                rows.append(row)
                n += 1
                continue

            row["converted"] = True

            try:
                _write_id3_mp3(dst, tags_easy, id3_src)
                row["tags_written"] = True
            except Exception as e:
                row["tags_written"] = False
                row["error"] = f"tag_write_failed: {e}"

            if not keep_originals:
                try:
                    src.unlink()
                except Exception:
                    pass

            row["status"] = "ok"
            rows.append(row)
            n += 1

        except Exception as e:
            row["status"] = "failed"
            row["error"] = str(e)
            rows.append(row)
            n += 1

    df_results = pd.DataFrame(rows)
    return df_results


In [3]:
audio_extensions = ["aiff", "aif", "wav"]

folder_in = txt_path

df_results = _mp3_0701_aiffwav2mp3tagname_GET_df_results(
    folder_in=folder_in,
    audio_extensions=audio_extensions,
    bitrate_kbps=320,
    sr_hz=44100,
    channels=2,
    n_start=1,
    overwrite=False,
    keep_originals=True
)

df_results


TQM | AIFF/WAV -> MP3 + tags + naming: 100%|██████████████████████████████████| 8/8 [00:17<00:00,  2.20s/file]


,n,src_path,src_ext,dst_path,dst_file,status,converted,tags_written,error
0,1,/Users/yerik/Desktop/trial mp3s/Barry Can't Sw...,.aiff,/Users/yerik/Desktop/trial mp3s/n1_TRkw_Barry_...,n1_TRkw_Barry_Can't_Swi-MB-4952988243103027-fr...,ok,True,True,None
1,2,/Users/yerik/Desktop/trial mp3s/Demuja - Perio...,.aiff,/Users/yerik/Desktop/trial mp3s/n2_TRkw_Demuja...,"n2_TRkw_Demuja_-_Period_Of_Time_(Album,_20.08....",ok,True,True,None
2,3,/Users/yerik/Desktop/trial mp3s/Disclosure - E...,.wav,/Users/yerik/Desktop/trial mp3s/n3_TRkw_Disclo...,n3_TRkw_Disclosure_-_Ecstasy_(Original_Mix)_2_...,ok,True,True,None
3,4,"/Users/yerik/Desktop/trial mp3s/Khadija, Marce...",.wav,/Users/yerik/Desktop/trial mp3s/n4_TRkw_Khadij...,"n4_TRkw_Khadija,_Marcel_Vogel,_Tim_Jules_-_Thi...",ok,True,True,None
4,5,/Users/yerik/Desktop/trial mp3s/Moi Je - Profi...,.wav,/Users/yerik/Desktop/trial mp3s/n5_TRkw_Moi_Je...,n5_TRkw_Moi_Je_-_Profite_(Kazy_Lambist_Remix)_...,ok,True,True,None
5,6,"/Users/yerik/Desktop/trial mp3s/Sio, Dwson - N...",.wav,"/Users/yerik/Desktop/trial mp3s/n6_TRkw_Sio,_D...","n6_TRkw_Sio,_Dwson_-_Nobody_Else_feat._Sio_(Ra...",ok,True,True,None
6,7,/Users/yerik/Desktop/trial mp3s/Tibi Dabo - Ko...,.wav,/Users/yerik/Desktop/trial mp3s/n7_TRkw_Tibi_D...,n7_TRkw_Tibi_Dabo_-_Komorebi_(Original_Mix)_2_...,ok,True,True,None
7,8,/Users/yerik/Desktop/trial mp3s/[Trun]_Truncat...,.wav,/Users/yerik/Desktop/trial mp3s/n8_TRkw_[Trun]...,n8_TRkw_[Trun]_Truncate_-_Culture_(Original_Mi...,ok,True,True,None
